In [ ]:
# ── Cell 1: Mount Drive and set working directory ─────────────────────────
from google.colab import drive  # type: ignore
import os, subprocess, sys
from pathlib import Path

drive.mount("/content/drive")

# ── Locate project root — check every common Drive layout ─────────────────
_CANDIDATES = [
    "/content/drive/MyDrive/PixelLab-StudyPal-RAG-DIP/smart-learning-assistant",
    "/content/drive/MyDrive/smart-learning-assistant/PixelLab-StudyPal-RAG-DIP/smart-learning-assistant",
    "/content/drive/MyDrive/PixelLab-StudyPal-RAG-DIP",
    "/content/drive/MyDrive/smart-learning-assistant",
]

DRIVE_PROJECT_ROOT = next(
    (p for p in _CANDIDATES
     if Path(p).is_dir() and Path(p, "app").is_dir()),  # must have app/ subdir
    None,
)

if DRIVE_PROJECT_ROOT is None:
    # Last-resort: ask user
    print("⚠️  Could not auto-detect project folder.")
    print("    Available top-level Drive folders:")
    for d in sorted(Path("/content/drive/MyDrive").iterdir()):
        if d.is_dir():
            print("       ", d)
    DRIVE_PROJECT_ROOT = input(
        "\nEnter FULL path to the smart-learning-assistant folder: "
    ).strip()
    if not Path(DRIVE_PROJECT_ROOT, "app").is_dir():
        sys.exit("❌ Path not valid — no app/ subdirectory found. Fix the path and re-run.")

print(f"Project root : {DRIVE_PROJECT_ROOT}")

# ── Pull latest code ───────────────────────────────────────────────────────
_repo_root = str(Path(DRIVE_PROJECT_ROOT).parent)
_pull = subprocess.run(
    ["git", "pull", "--ff-only"],
    cwd=_repo_root, capture_output=True, text=True,
)
_msg = (_pull.stdout + _pull.stderr).strip() or "Already up to date."
print("git pull:", _msg)

os.chdir(DRIVE_PROJECT_ROOT)
print(f"Working dir  : {os.getcwd()}")
print("Files        :", sorted(os.listdir(".")))


In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────────────────
# ragas 0.1.x used langchain_core.pydantic_v1, removed in langchain-core 0.3.
# That forced numpy→1.26.4, which breaks Colab's pandas (compiled for numpy 2).
# ragas>=0.2.0 dropped pydantic_v1 → works with langchain-core 0.3.x + numpy 2.x.
%pip install -q \
    "ragas>=0.2.0,<0.3.0" \
    "langchain-groq>=0.2.0" \
    "langchain-community>=0.3.0" \
    "langchain-huggingface>=0.1.0" \
    "datasets>=2.18" \
    "sentence-transformers" \
    "python-dotenv"

print("\n✅ Packages installed. Restart runtime if Colab prompts you.")


In [ ]:
# ── Cell 3: Set Groq API key (free judge LLM for RAGAS) ───────────────────
# Get a free key at https://console.groq.com/keys (sign in with GitHub/Google)
import os
from getpass import getpass

groq_key = getpass("Enter your Groq API key (gsk_...): ")
os.environ["GROQ_API_KEY"] = groq_key

# Quick sanity check
from langchain_groq import ChatGroq  # type: ignore
_test = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
_resp = _test.invoke("Reply with the single word OK.")
_text = _resp.content if isinstance(_resp.content, str) else (_resp.content[0] if _resp.content else "OK")
print(f"✅ Groq key valid. Test response: {_text}")
print("   RAGAS judge  : llama-3.1-8b-instant (500K TPD)")
print("   Embeddings   : all-MiniLM-L6-v2 (local, sentence-transformers)")
print("   RunConfig    : max_workers=1 (sequential) · timeout=3600s")


In [ ]:
# ── Cell 4: Load intermediate JSON ────────────────────────────────────────
import json
from pathlib import Path

INTERMEDIATE_PATH = Path("data/eval_intermediate.json")

if not INTERMEDIATE_PATH.exists():
    # Fallback: allow manual upload from local machine
    from google.colab import files # type: ignore
    print("eval_intermediate.json not found in Drive project folder.")
    print("Uploading from your local machine instead...")
    uploaded = files.upload()          # shows a file-picker in Colab
    INTERMEDIATE_PATH = Path(list(uploaded.keys())[0])

with open(INTERMEDIATE_PATH, encoding="utf-8") as f:
    intermediate = json.load(f)

n_questions  = len(intermediate["questions"])
n_guardrail  = len(intermediate.get("guardrail_results", []))
collected_at = intermediate.get("collected_at", "unknown")

print(f"✅ Loaded intermediate data")
print(f"   DIP questions  : {n_questions}")
print(f"   Guardrail tests: {n_guardrail}")
print(f"   Collected at   : {collected_at}")

# Preview first question
print(f"\n   Q1: {intermediate['questions'][0]}")
print(f"   A1: {intermediate['answers'][0][:120]}...")

In [ ]:
# ── Cell 5: RAGAS scoring — fully self-contained, no Drive imports ─────────
# ragas 0.2.x column names: user_input / response / retrieved_contexts / reference
#
# ROOT CAUSE OF ALL TIMEOUTS:
#   ragas uses asyncio.gather internally → ALL sub-calls fire concurrently,
#   regardless of max_workers=1.  faithfulness alone does 4-6 async LLM calls
#   per question → burst hits Groq's 6000 TPM → retry-after=60 inside a job →
#   timeout fires before retry succeeds.
#
# THE ONLY RELIABLE FIX:
#   InMemoryRateLimiter on ChatGroq gates every sub-call at the source.
#   0.1 req/s = 6 req/min × ~800 tok/req ≈ 4800 tok/min → under 6000 TPM.
import sys, os, time, warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np          # type: ignore
import pandas as pd         # type: ignore

# Suppress noisy-but-harmless unauthenticated HF Hub warning (public models fine)
warnings.filterwarnings("ignore", category=UserWarning, module="huggingface_hub")

sys.path.insert(0, str(Path.cwd()))

from datasets import Dataset                              # type: ignore
from ragas import evaluate                                # type: ignore
try:
    from ragas import RunConfig                           # type: ignore
except ImportError:
    from ragas.run_config import RunConfig                # type: ignore

from ragas.metrics import (                               # type: ignore
    answer_relevancy, context_precision,
    context_recall, faithfulness,
)
from ragas.llms import LangchainLLMWrapper                # type: ignore
from ragas.embeddings import LangchainEmbeddingsWrapper   # type: ignore
from langchain_groq import ChatGroq                       # type: ignore
try:
    from langchain_huggingface import HuggingFaceEmbeddings   # type: ignore
except ImportError:
    from langchain_community.embeddings import HuggingFaceEmbeddings  # type: ignore

# ── Rate limiter — 0.1 req/s = 6 calls/min ≈ 4800 tok/min (under 6K TPM) ─
_rate_limiter   = None
_between_metric = 30    # seconds to sleep between metrics (normal path)

try:
    from langchain_core.rate_limiters import InMemoryRateLimiter  # type: ignore
    _rate_limiter = InMemoryRateLimiter(
        requests_per_second=0.1,      # 1 call per 10 s → ~4800 tok/min
        check_every_n_seconds=0.05,
        max_bucket_size=1,
    )
    print("✅ Rate limiter: 0.1 req/s (every LLM sub-call throttled automatically)")
except ImportError:
    _between_metric = 90
    print("⚠️  InMemoryRateLimiter not available — manual 90 s sleep between metrics.")
    print(f"   → Proceeding anyway — between-metric sleep extended to {_between_metric}s.")

# ── Preamble-stripping wrapper ─────────────────────────────────────────────
# llama-3.1-8b-instant sometimes emits a chain-of-thought paragraph BEFORE
# the JSON block.  ragas reads the output via LangchainLLMWrapper.agenerate_text()
# (and generate_text()) which return List[str].  Stripping THERE means the
# JSON parser never sees the preamble, regardless of what ChatGroq returns.
#
# WHY NOT patch ChatGroq or ChatGeneration.text:
#   • response_format:json_object requires "json" in the prompt → ragas prompts
#     don't have it → model gets confused → faithfulness 0.352→0.227 (WORSE).
#   • ChatGeneration.text is a @property backed by message.content in pydantic v2;
#     assigning gen.text=c creates an instance attr shadowed by the descriptor —
#     the property still returns the original preamble-laden content.
#
# CORRECT FIX: override agenerate_text / generate_text in LangchainLLMWrapper.
class _CleaningLLMWrapper(LangchainLLMWrapper):
    """LangchainLLMWrapper that strips pre-JSON preamble from every LLM response.

    agenerate_text / generate_text return LLMResult (NOT List[str]).
    We clean each generation's .text field using pydantic model_copy() —
    which bypasses __init__ validators — so the ragas JSON parser always
    receives clean JSON regardless of pydantic version.
    """

    @staticmethod
    def _strip(text: str) -> str:
        idx = text.find('{')
        return text[idx:] if idx > 0 else text

    def _clean_result(self, result):
        """Return a new LLMResult with pre-JSON preambles stripped."""
        new_gens, changed = [], False
        for gen_list in result.generations:
            new_list = []
            for gen in gen_list:
                t = getattr(gen, 'text', '') or ''
                if isinstance(t, str) and t.find('{') > 0:
                    c = self._strip(t)
                    # model_copy (pydantic v2) bypasses validators → text stays clean
                    try:
                        gen = gen.model_copy(update={'text': c})
                    except AttributeError:
                        try:
                            gen = gen.copy(update={'text': c})   # pydantic v1
                        except Exception:
                            try:
                                gen.text = c
                            except Exception:
                                object.__setattr__(gen, 'text', c)
                    changed = True
                new_list.append(gen)
            new_gens.append(new_list)
        if not changed:
            return result
        try:
            return result.model_copy(update={'generations': new_gens})
        except AttributeError:
            try:
                return result.copy(update={'generations': new_gens})
            except Exception:
                result.generations = new_gens
                return result

    async def agenerate_text(self, prompt, n=1, temperature=1e-8,
                              stop=None, callbacks=None):
        result = await super().agenerate_text(
            prompt, n=n, temperature=temperature, stop=stop, callbacks=callbacks
        )
        return self._clean_result(result)

    def generate_text(self, prompt, n=1, temperature=1e-8,
                      stop=None, callbacks=None):
        result = super().generate_text(
            prompt, n=n, temperature=temperature, stop=stop, callbacks=callbacks
        )
        return self._clean_result(result)


# ── Judge LLM ─────────────────────────────────────────────────────────────
# NO response_format:json_object — see comment above.
_groq_kwargs = dict(
    model="llama-3.1-8b-instant",
    temperature=0,
    max_tokens=4096,
)
if _rate_limiter is not None:
    _groq_kwargs["rate_limiter"] = _rate_limiter

_raw_llm  = ChatGroq(**_groq_kwargs)
ragas_llm = _CleaningLLMWrapper(_raw_llm)
ragas_emb = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
)

# ── Set LLM/embeddings on each metric (ragas 0.2.x) ──────────────────────
_METRICS     = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
_TARGET      = 0.7
_metric_objs = [faithfulness, answer_relevancy, context_precision, context_recall]
for _m in _metric_objs:
    _m.llm = ragas_llm
    if hasattr(_m, "embeddings"):
        _m.embeddings = ragas_emb

# ── Load intermediate data ─────────────────────────────────────────────────
import json
_inter_path = Path("data/eval_intermediate.json")
if not _inter_path.exists():
    raise FileNotFoundError(
        "data/eval_intermediate.json not found — run Cell 3 first, "
        "or re-run the collection cell and re-save."
    )
with open(_inter_path, encoding="utf-8") as _f:
    intermediate = json.load(_f)

print(f"✅ Loaded {len(intermediate.get('questions', []))} questions from {_inter_path}")

# ── Truncate context chunks ────────────────────────────────────────────────
_MAX_CHUNK_CHARS = 400
_contexts_truncated = [
    [c[:_MAX_CHUNK_CHARS] for c in ctx_list]
    for ctx_list in intermediate["contexts"]
]
_ctx_total = sum(len(c) for ctx in intermediate["contexts"] for c in ctx)
_ctx_after = sum(len(c) for ctx in _contexts_truncated for c in ctx)
print(f"Context truncation: {_ctx_total:,} → {_ctx_after:,} chars "
      f"({(100*_ctx_after//_ctx_total) if _ctx_total else 100}% of original)")

# ── Truncate answers ──────────────────────────────────────────────────────
_MAX_ANS_CHARS     = 500
_answers_truncated = [a[:_MAX_ANS_CHARS] for a in intermediate["answers"]]
_ans_total         = sum(len(a) for a in intermediate["answers"])
_ans_after         = sum(len(a) for a in _answers_truncated)
print(f"Answer truncation:  {_ans_total:,} → {_ans_after:,} chars "
      f"({(100*_ans_after//_ans_total) if _ans_total else 100}% of original)")

# ── Reference for context_precision / context_recall ──────────────────────
# IMPORTANT: reference must be the *expected answer* text, NOT the question.
# Using questions gave context_precision=0.033 because the LLM judged chunks
# against "What is X?" instead of against the actual answer content.
_gt = intermediate.get("ground_truths", [])
_n  = len(intermediate["questions"])
if _gt:
    _reference = [(g[0] if isinstance(g, list) else str(g)) for g in _gt][:_n]
    if len(_reference) < _n:
        _reference += list(_answers_truncated)[len(_reference):]
    print(f"ℹ️  Using {min(len(_gt), _n)} ground-truth references from JSON")
else:
    _reference = list(_answers_truncated)
    print("ℹ️  No ground_truths in JSON — using system answers as reference")
    print("   context_recall = answer grounding · context_precision = chunk utility")

dataset = Dataset.from_dict({
    "user_input":         intermediate["questions"],
    "response":           _answers_truncated,
    "retrieved_contexts": _contexts_truncated,
    "reference":          _reference,
})

# ── RunConfig ─────────────────────────────────────────────────────────────
# timeout=3600: asyncio.gather stamps t=0 on ALL sub-tasks simultaneously.
# Job[N] waits N×10 s in the rate-limiter queue before its first call.
# 3600 s gives every job up to 1 h from dispatch — absorbs any retry storm.
run_cfg = RunConfig(timeout=3600, max_workers=1)

# ── Run ONE metric at a time + sleep between them ─────────────────────────
print(f"\nRunning 4 metrics × {_n} questions  (rate-limited · one metric at a time)")
print(f"⏱  Expected: ~35–45 min.  Do NOT interrupt — rate limiter is working!\n")

_score_columns: dict = {}

for _i, (_mobj, _mname) in enumerate(zip(_metric_objs, _METRICS)):
    _t0 = time.time()
    print(f"[{_i+1}/4] {_mname} …", flush=True)
    _r = evaluate(
        dataset,
        metrics=[_mobj],
        llm=ragas_llm,
        embeddings=ragas_emb,
        run_config=run_cfg,
    )
    _df_m: pd.DataFrame = (
        _r.to_pandas() if hasattr(_r, "to_pandas") else pd.DataFrame(dict(_r))
    )
    _elapsed = time.time() - _t0

    if _mname in _df_m.columns:
        _score_columns[_mname] = _df_m[_mname].tolist()
        _avg    = float(_df_m[_mname].mean())
        _status = "✅" if _avg >= _TARGET else "❌"
        print(f"    {_status} mean={_avg:.3f}  ({_elapsed/60:.1f} min)")
    else:
        _numeric = [c for c in _df_m.columns
                    if c not in ("user_input","response","retrieved_contexts","reference")]
        if _numeric:
            _score_columns[_mname] = _df_m[_numeric[0]].tolist()
            _avg    = float(_df_m[_numeric[0]].mean())
            _status = "✅" if _avg >= _TARGET else "❌"
            print(f"    {_status} score from column '{_numeric[0]}' mean={_avg:.3f}  ({_elapsed/60:.1f} min)")
        else:
            _score_columns[_mname] = [float("nan")] * len(dataset)
            print(f"    ⚠️  no score column found — NaN stored  ({_elapsed/60:.1f} min)")

    if _i < 3:
        print(f"   ↳ sleeping {_between_metric}s (TPM window flush) …")
        time.sleep(_between_metric)

# ── Assemble final DataFrame ───────────────────────────────────────────────
ragas_df = pd.DataFrame({
    "user_input": dataset["user_input"],
    "response":   dataset["response"],
    **_score_columns,
})

# Save CSV immediately so it's available even if report generation fails
_csv_path = Path("data/ragas_scores.csv")
_csv_path.parent.mkdir(parents=True, exist_ok=True)
ragas_df.to_csv(_csv_path, index=False)
print(f"\n✅ Scores saved to {_csv_path}")

# ── Pretty-print scores ───────────────────────────────────────────────────
print("\n=== RAGAS Scores ===")
print(f"{'Metric':<25} {'Score':>7}  {'Target':>7}  Status")
print("-" * 52)
for m in _METRICS:
    if m in ragas_df.columns:
        score  = float(ragas_df[m].mean())
        status = "✅ PASS" if score >= _TARGET else "❌ FAIL"
        print(f"{m:<25} {score:>7.3f}  {_TARGET:>7.1f}  {status}")

# ── Build markdown report ─────────────────────────────────────────────────
def _build_report(df: pd.DataFrame,
                  latencies: list,
                  guardrail_results: list,
                  topic_map: list | None = None) -> str:
    ts   = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
    lats = sorted(latencies) if latencies else []
    lines = [
        "# Evaluation Report — DIP AI Tutor",
        f"\n_Generated: {ts}_\n",
        "\n## Overall RAGAS Scores\n",
        "| Metric | Score | Target | Status |",
        "|--------|-------|--------|--------|",
    ]
    for m in _METRICS:
        if m in df.columns:
            s = float(df[m].mean())
            lines.append(
                f"| {m} | {s:.3f} | {_TARGET:.1f} | {'✅' if s >= _TARGET else '❌'} |"
            )

    if topic_map and all(c in df.columns for c in ["faithfulness", "answer_relevancy"]):
        _df2 = df.copy()
        _df2["topic"] = topic_map[:len(_df2)]
        lines += [
            "\n## Per-Topic Breakdown\n",
            "| Topic | Faithfulness | Answer Relevancy |",
            "|-------|-------------|-----------------|",
        ]
        for topic, grp in _df2.groupby("topic"):
            lines.append(
                f"| {str(topic)[:60]} | {float(grp['faithfulness'].mean()):.3f} "
                f"| {float(grp['answer_relevancy'].mean()):.3f} |"
            )

    if lats:
        p95 = float(np.percentile(lats, 95))
        lines += [
            "\n## Latency Analysis\n",
            f"- **Mean**: {float(np.mean(lats)):.2f}s",
            f"- **p50**: {float(np.percentile(lats, 50)):.2f}s",
            f"- **p95**: {p95:.2f}s" + (" ⚠️ exceeds 5 s target" if p95 > 5 else " ✅"),
        ]

    if guardrail_results:
        lines += [
            "\n## Guardrail Test Results\n",
            "| Question | Status | Answer Preview |",
            "|----------|--------|----------------|",
        ]
        for g in guardrail_results:
            lines.append(
                f"| {str(g.get('question', ''))[:60]} "
                f"| {g.get('status', '?')} "
                f"| {g.get('answer', '')[:80].replace(chr(10), ' ')} |"
            )

    lines += ["\n## Failed Cases (score < 0.7)\n"]
    any_bad = False
    for idx, (_, row) in enumerate(df.iterrows()):
        bad = [m for m in _METRICS if m in row and float(row[m]) < _TARGET]
        if bad:
            any_bad = True
            lines.append(f"**Q{idx+1}**: {str(row.get('user_input', ''))[:80]}")
            for m in bad:
                lines.append(f"  - `{m}`: {float(row[m]):.3f}")
    if not any_bad:
        lines.append("_All questions meet the 0.7 threshold._ ✅")

    return "\n".join(lines)

# ── Generate and save report ───────────────────────────────────────────────
print("\nGenerating markdown report …")
report_md = _build_report(
    ragas_df,
    latencies=intermediate.get("latencies", []),
    guardrail_results=intermediate.get("guardrail_results", []),
    topic_map=intermediate.get("topic_map"),
)
Path("evaluation_report.md").write_text(report_md, encoding="utf-8")
print("✅ Report generated and saved to evaluation_report.md")


In [ ]:
# ── Cell 6: Save report to Drive and offer local download ─────────────────
from pathlib import Path
from google.colab import files as colab_files # type: ignore

REPORT_PATH = Path("evaluation_report.md")

# The report was already written to evaluation_report.md by generate_report()
if REPORT_PATH.exists():
    print(f"📄 Report saved at: {REPORT_PATH.resolve()}")
    print(f"   Size: {REPORT_PATH.stat().st_size:,} bytes")

    # Download to local machine
    print("\nDownloading evaluation_report.md to your local machine...")
    colab_files.download(str(REPORT_PATH))

    # Also save RAGAS DataFrame as CSV
    csv_path = Path("data/ragas_scores.csv")
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    ragas_df.to_csv(csv_path, index=False)
    colab_files.download(str(csv_path))
    print("✅ Downloaded evaluation_report.md and ragas_scores.csv")
else:
    print("⚠️  Report file not found. Check for errors in Cell 5.")

# Print the report inline too
print("\n" + "=" * 70)
print(report_md)